# Direct Oλ gene associations and preranked enrichment
Reuse the harmonized cell-bootstrap pseudobulk particles over the shared STRING genes. Replace the spectral target with the bounded direct-scale asymptote particles. In each of 4,000 draws, the original estimator selects a whole-cohort pseudobulk realization, draws a spectral particle per patient, and computes patient-bootstrap-weighted partial Spearman correlations after dataset adjustment.

Rank every eligible gene by posterior mean correlation divided by posterior SD. Test Hallmark and Reactome against this ranking with the exact MSigDB memberships saved by the prior particle analysis. Positive NES points toward higher direct Oλ. Each completed stage can be reused on rerun.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
PB = DATA/'string_pseudobulk_particles'
PSEUDOBULK = PB/'patient_pseudobulk_particles_harmonized.npz'
METADATA = PB/'patient_metadata_dataset_only.csv'
DIRECT = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1'
ESTIMATES = DIRECT/'patient_olambda_asymptotes.csv'
SPECTRAL = DIRECT/'patient_olambda_particles.csv'
PREVIOUS = DATA/'bayesian_Olambda_gene_associations_dataset_adjusted_pseudobulk_particles'
REFERENCE = PREVIOUS/'fgsea'
BASE = DATA/'bayesian_direct_Olambda_gene_associations_dataset_adjusted_pseudobulk_particles'
DRAWS, SEED, N_PERM_SIMPLE = 4000, 0, 1_000_000

In [ ]:
REVISION = '6f662c535f0826c9be8dfde12fd2b13f80091dbe'
repo = Path(tempfile.mkdtemp(prefix='direct-gsea-'))/'repo'
subprocess.run(['git','clone','--quiet','--single-branch','--branch','direct-olambda',
                'https://github.com/adamnik/CAR-T-spectral-organization.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--quiet','--detach',REVISION],check=True)
scripts=repo/'gene_enrichment'
def run(cmd):
    print(' '.join(map(str,cmd)),flush=True)
    env=dict(os.environ,OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
    with subprocess.Popen(list(map(str,cmd)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=env,text=True,bufsize=1) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
        if process.wait(): raise subprocess.CalledProcessError(process.returncode,cmd)

In [ ]:
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
refs=[REFERENCE/'fgsea_collection_versions.csv',*[REFERENCE/f'membership_{n}.csv' for n in ('Hallmark','Reactome')]]
files=[PSEUDOBULK,METADATA,ESTIMATES,SPECTRAL,*refs]
for p in files:
    if not p.is_file(): raise FileNotFoundError(p)
keys=['Dataset','sample_id']
est=pd.read_csv(ESTIMATES).set_index(keys,verify_integrity=True).sort_index()
particles=pd.read_csv(SPECTRAL).set_index(keys,verify_integrity=True).sort_index()
assert est.index.equals(particles.index) and len(est)==163 and est.index.get_level_values(0).nunique()==8
assert est.model.eq('bounded_direct_no_asymptote_floor').all() and est.olambda_scale.eq('reciprocal').all()
assert list(particles)==[f'particle_{i:02d}' for i in range(1,17)] and ((particles>0)&(particles<1)).all().all()
meta=pd.read_csv(METADATA).set_index(keys,verify_integrity=True)
assert est.index.difference(meta.index).empty
with np.load(PSEUDOBULK,allow_pickle=False) as a:
    ids=pd.MultiIndex.from_arrays([a['datasets'].astype(str),a['patient_ids'].astype(str)],names=keys)
    assert ids.is_unique and est.index.difference(ids).empty
    rows=ids.get_indexer(est.index)
    assert (a['n_cells'][rows]>=1000).all()
    weights=a['particle_weights'][rows]
    assert weights.shape==(163,16) and np.allclose(weights,1/16)
    genes=a['feature_names'].astype(str)
    assert pd.Index(genes).is_unique
versions=pd.read_csv(refs[0]).set_index('collection',verify_integrity=True)
assert set(versions.index)=={'Hallmark','Reactome'}
print(f'{len(est)} patients | 8 datasets | {len(genes):,} shared STRING genes before filtering')
display(versions)
config=dict(revision=REVISION,draws=DRAWS,seed=SEED,min_cells=1000,min_size=15,max_size=500,
            simple_permutations=N_PERM_SIMPLE,adjustment='dataset',input_sha256={p.name:sha(p) for p in files})
tag=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:12]
OUTPUT=BASE/tag
OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/'run_config.json').write_text(json.dumps(config,indent=2))
print('Output:',OUTPUT)

## Dataset-adjusted gene correlations

In [ ]:
association_files=['olambda_gene_associations.csv','olambda_gene_GSEA_rank.rnk','olambda_gene_GSEA_rank.csv',
                   'analysis_cohort.csv','gene_association_manifest.json']
if not all((OUTPUT/name).exists() for name in association_files):
    with tempfile.TemporaryDirectory(dir=OUTPUT) as staging:
        stage=Path(staging)
        run([sys.executable,'-u',scripts/'estimate_gene_olambda_correlations_particles.py',
             PSEUDOBULK,SPECTRAL,METADATA,stage,'--minimum-cells',1000,'--draws',DRAWS,'--random-state',SEED])
        for f in sorted(stage.iterdir(),key=lambda f:f.name=='gene_association_manifest.json'):
            f.replace(OUTPUT/f.name)
fit=pd.read_csv(OUTPUT/'olambda_gene_associations.csv')
used=pd.read_csv(OUTPUT/'analysis_cohort.csv').set_index(keys,verify_integrity=True)
assert used.index.sort_values().equals(est.index) and fit.adjustment.eq('dataset_fixed_effects').all()
prior_table=PREVIOUS/'olambda_gene_associations.csv'
if prior_table.is_file():
    old=pd.read_csv(prior_table,usecols=['gene'])
    assert set(old.gene)==set(fit.gene), 'Eligible gene universe differs from the prior particle run'
assert fit.gene.is_unique and np.isfinite(fit.posterior_standardized_effect).all()
print('Eligible genes ranked:',len(fit))
display(fit.iloc[np.r_[0:5,-5:0]][['gene','rho_median','rho_q025','rho_q975','posterior_standardized_effect']])

## Hallmark and Reactome enrichment
Use pathways with 15–500 genes represented in the ranking, multilevel fgsea with one million preliminary simple permutations, and BH adjustment within each collection. The saved membership CSVs supply the original pathway definitions.

In [ ]:
FGSEA=OUTPUT/'fgsea'
expected=['fgsea_all_collections.csv','fgsea_Hallmark.csv','fgsea_Reactome.csv','membership_Hallmark.csv',
          'membership_Reactome.csv','fgsea_collection_versions.csv','R_session_info.txt']
if not all((FGSEA/name).is_file() for name in expected):
    install='''
options(repos=c(CRAN="https://cloud.r-project.org"))
if (!requireNamespace("data.table", quietly=TRUE)) install.packages("data.table")
if (!requireNamespace("fgsea", quietly=TRUE)) {
    if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
    BiocManager::install("fgsea", ask=FALSE, update=FALSE)
}
stopifnot(requireNamespace("data.table",quietly=TRUE), requireNamespace("fgsea",quietly=TRUE))
'''
    run(['Rscript','-e',install])
    FGSEA.mkdir(exist_ok=True)
    with tempfile.TemporaryDirectory(dir=OUTPUT) as staging:
        stage=Path(staging)
        run(['Rscript',scripts/'run_fgsea_rankings.R',OUTPUT/'olambda_gene_GSEA_rank.rnk',
             stage,15,500,N_PERM_SIMPLE,REFERENCE,SEED])
        assert all((stage/name).is_file() for name in expected)
        for name in expected: (stage/name).replace(FGSEA/name)
results=pd.read_csv(FGSEA/'fgsea_all_collections.csv')
actual=pd.read_csv(FGSEA/'fgsea_collection_versions.csv').set_index('collection')
assert actual.msigdb_version.sort_index().equals(versions.msigdb_version.sort_index())
assert actual.simple_permutations.eq(N_PERM_SIMPLE).all()
summary=results.assign(positive=(results.padj<.05)&(results.NES>0),
                       negative=(results.padj<.05)&(results.NES<0),
                       unresolved=results[['NES','pval','padj']].isna().any(axis=1)).groupby('collection').agg(
                       tested=('pathway','size'),positive=('positive','sum'),negative=('negative','sum'),
                       unresolved=('unresolved','sum'))
display(summary)
summary.to_csv(FGSEA/'enrichment_summary.csv')
print('Gene results:',OUTPUT,'\nEnrichment:',FGSEA)